# GPU-MLMC congestion: Colab reproduction of key results

This notebook reproduces two headline results of the IEEE Access article
*GPU-Accelerated Uncertainty Quantification for Network Congestion: Coupled SDE Simulation and a
Systematic Comparison of Rare-Event Estimators*, using the scripts in this repository:

1. **Rare-event comparison (Table 5).** Unbiased importance sampling with the Freidlin--Wentzell-optimal
   tilt is the most efficient estimator of the overflow probability $P(Q_{\max}\ge B)$ at every threshold,
   checked against the exact Chapman--Kolmogorov reference.
2. **Classical variance reduction (Table 6).** At the calibrated noise level $\sigma=0.433$, antithetic
   variates and control variates cut accuracy-matched work for the mean queue occupancy by roughly
   $30\times$ and $16\times$.

Both runs use fewer seeds and smaller batches than the paper so they finish in under 10 minutes on a
free Colab runtime, so the numbers agree with the paper to within sampling noise rather than exactly.
The full-size runs are `scripts/run_rare_event_comparators.py` and `scripts/run_qmc_baseline.py`
with their default arguments.

In [ ]:
# Cell 1: get the code (Colab already provides torch, numpy and scipy)
import os, sys, subprocess
if not os.path.exists('gpu-mlmc-congestion'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/pdwi2020/gpu-mlmc-congestion.git'], check=True)
os.chdir('gpu-mlmc-congestion')
import torch, numpy, scipy
print('Python', sys.version.split()[0], '| torch', torch.__version__,
      '| numpy', numpy.__version__, '| scipy', scipy.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Cell 2: rare-event estimator comparison (paper Table 5), 5 seeds, about 10-30 s
import subprocess, time
t0 = time.time()
subprocess.run([sys.executable, 'scripts/run_rare_event_comparators.py',
                '--seeds', '0', '1', '2', '3', '4', '--B', '15', '20', '25',
                '--out', 'results/colab/rare_event'], check=True)
print(f'rare-event comparison finished in {time.time() - t0:.0f} s')

In [ ]:
# Cell 3: classical variance reduction at calibrated sigma (paper Table 6), about 3-6 min
t0 = time.time()
subprocess.run([sys.executable, 'scripts/run_qmc_baseline.py',
                '--arms', 'mc', 'antithetic', 'cv', '--sigmas', '0.433',
                '--ref-samples', '4000', '--var-samples', '4000', '--cv-pilot', '512',
                '--out', 'results/colab/varred'], check=True)
print(f'variance-reduction baseline finished in {time.time() - t0:.0f} s')

In [ ]:
# Cell 4: compare with the paper
import csv

def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(line for line in f if not line.startswith('#')))

paper_exact = {15: 6.3e-4, 20: 4.2e-6, 25: 8.3e-9}
paper_eff = {15: {'gpu_mlmc_is': 2.3e3, 'ams': 4.3e3},
             20: {'gpu_mlmc_is': 3.5e3, 'ams': 4.3e4},
             25: {'gpu_mlmc_is': 1.1e4, 'ams': 2.2e5}}

rows = read_csv('results/colab/rare_event/summary.csv')
print('Rare-event overflow probability (efficiency = relRMSE^2 x work, lower is better)')
for B in (15, 20, 25):
    sub = [r for r in rows if float(r['B']) == B]
    eff = {r['method']: float(r['work_norm_efficiency']) for r in sub}
    best = min(eff, key=eff.get)
    est = next(float(r['mean_estimate']) for r in sub if r['method'] == 'gpu_mlmc_is')
    print(f'  B={B}: exact P={float(sub[0]["ref"]):.2e} (paper {paper_exact[B]:.1e}), '
          f'IS estimate {est:.2e}; IS eff {eff["gpu_mlmc_is"]:.1e} (paper {paper_eff[B]["gpu_mlmc_is"]:.1e}), '
          f'AMS eff {eff["ams"]:.1e} (paper {paper_eff[B]["ams"]:.1e}); best = {best}')

vr = {r['method']: r for r in read_csv('results/colab/varred/summary.csv')}
print()
print('Accuracy-matched work reduction vs plain MC at eps = 5e-3 (sigma = 0.433)')
for m, paper in (('antithetic', 30.3), ('cv', 16.4)):
    print(f'  {m:10s}: {float(vr[m]["work_ratio_vs_mc_eps_0.005"]):5.1f}x  (paper {paper}x)')